# View Purge Behavior — full gate matrix

> **New here? Read this first.** A view drop's outcome depends on FOUR things, and
> this notebook maps how they interact — including isolating which one causes the
> `403` that an earlier session saw.

## The four dimensions

```
purge-view-metadata-on-drop   catalog config — should the drop delete the metadata file?
drop-with-purge.enabled       catalog config — is purge allowed at all?
?purgeRequested=              request param  — true / false / absent
CATALOG_MANAGE_CONTENT        privilege      — does the principal have purge rights?
```

In our no-STS environment, even a successful purging drop **orphans** the metadata
file (the async cleanup task fails — Apache Polaris #379, see the test report).

## The 8-case matrix

```
C1  meta=true  purge=false  grant=NO   req=absent   ← the original "403" scenario
C2  meta=true  purge=false  grant=YES  req=absent   isolates the PRIVILEGE gate
C3  meta=true  purge=true   grant=NO   req=absent   isolates the CONFIG gate
C4  meta=true  purge=true   grant=YES  req=absent   everything enabled
C5  meta=true  purge=true   grant=YES  req=false    explicit purgeRequested=false
C6  meta=false purge=true   grant=YES  req=true     param vs config conflict
C7  meta=false purge=true   grant=YES  req=absent   metadata-only baseline
C8  drop an already-dropped view                    404 idempotency
```

The final cell derives, from the observed status codes, **which gate produces the
403** (privilege vs config) — answering the open question rather than assuming it.

## Setup note (important)

```
A grant can't be cleanly REVOKED mid-test, so the runner uses TWO catalogs:
  view-test-grant     — CATALOG_MANAGE_CONTENT granted
  view-test-nogrant   — NOT granted
Each case runs against the right one. Both are cleaned up (verified) at the end.
```

## Prerequisites & how to run

```
Files:   polaris_test_utils.py, minio_rest.py, config/  (same folder)
Packages: pip install requests pyyaml opensearch-py
Run:     setup cell (🟢/🔴 banner) → force_delete_catalog def → matrix setup →
         C1–C8 → truth table → cleanup.
Switch:  change init_env("dev") → init_env("prod") in setup, re-run that cell.

⚠️  Storage via MinIO REST API (minio_rest.py) — NO s3fs. No hardcoded creds.
```

---


In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────
# Environment switch HERE. Banner shows 🟢 DEV / 🔴 PROD. Change "dev" → "prod"
# and re-run THIS cell to switch (no kernel restart needed).
from polaris_test_utils import *
init_env("dev")

import uuid, time
tok = root_token()

# Map friendly kwarg names → the EXACT Polaris property keys.
# (Property names mix hyphens AND dots — a blind _→- replace produces the wrong
#  key, e.g. "drop-with-purge-enabled" instead of "drop-with-purge.enabled",
#  which Polaris rejects with 400. That silent failure caused earlier wrong results.)
_CONFIG_KEYS = {
    "drop_with_purge_enabled":       "polaris.config.drop-with-purge.enabled",
    "purge_view_metadata_on_drop":   "polaris.config.purge-view-metadata-on-drop",
}

def set_catalog_config(catalog, _retries=5, **flags):
    """Merge config flags into a catalog AND VERIFY they applied.
    Uses the exact property keys, checks the PUT response, retries on 409
    entityVersion conflict, and confirms the values changed by reading back."""
    want = {}
    for k, v in flags.items():
        if k not in _CONFIG_KEYS:
            raise KeyError(f"unknown config flag '{k}'. Known: {list(_CONFIG_KEYS)}")
        want[_CONFIG_KEYS[k]] = v
    for attempt in range(_retries):
        cat = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json()
        merged = {**cat.get("properties", {}), **want}
        r = requests.put(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok),
                         json={"currentEntityVersion": cat["entityVersion"], "properties": merged})
        if r.status_code in (200, 204):
            now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
            if all(now.get(k) == v for k, v in want.items()):
                return True
        elif r.status_code == 409:
            time.sleep(0.2); continue
        else:
            raise RuntimeError(f"set_catalog_config PUT failed [{r.status_code}]: {r.text[:200]}")
    now = requests.get(f"{BASE_MGMT}/catalogs/{catalog}", headers=h(tok)).json().get("properties", {})
    if not all(now.get(k) == v for k, v in want.items()):
        raise RuntimeError(f"set_catalog_config could not apply {want} to {catalog} "
                           f"(now: { {k: now.get(k) for k in want} })")
    return True

print("✅ setup ready (correct property keys + verified writes)")


🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 DEV   POLARIS_ENV=dev   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
✅ setup ready (correct property keys + verified writes)


In [2]:
# HARDENED cleanup — removes a catalog completely and VERIFIES it's gone.
# Key fixes:
#  • drop views/tables WITHOUT ?purgeRequested (avoids the purge-coupling that
#    can make a view drop fail and leave the namespace non-empty)
#  • set purge-view-metadata=false BEFORE dropping views (so drop is metadata-only)
#  • verify namespace is empty before deleting it; verify catalog gone at the end
# A stale catalog with old config flags is what corrupted earlier test runs.

def _ns_name(ns):
    return ns[0] if isinstance(ns, list) else ns

def force_delete_catalog(cat):
    if requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code == 404:
        return True
    # make teardown safe: purge OFF for views (metadata-only drop), purge ON allowed
    set_catalog_config(cat, drop_with_purge_enabled="true", purge_view_metadata_on_drop="false")
    grant_privilege(catalog=cat, cr="catalog_admin", privilege="CATALOG_MANAGE_CONTENT", token=tok)

    nss = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
    # deepest namespaces first
    nss = sorted(nss, key=lambda n: len(n if isinstance(n, list) else [n]), reverse=True)
    for ns in nss:
        nsn = _ns_name(ns)
        # drop all tables (plain drop — no purge; storage swept separately)
        for t in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", []):
            rt = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables/{t['name']}", headers=h(tok))
            if rt.status_code not in (200,204):
                print(f"   ⚠️ table {t['name']} drop: {rt.status_code} {rt.text[:80]}")
        # drop all views (plain drop — no purge)
        for v in requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", []):
            rv = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views/{v['name']}", headers=h(tok))
            if rv.status_code not in (200,204):
                print(f"   ⚠️ view {v['name']} drop: {rv.status_code} {rv.text[:80]}")
        # confirm empty, then delete namespace
        tleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/tables", headers=h(tok)).json().get("identifiers", [])
        vleft = requests.get(f"{BASE_CAT}/{cat}/namespaces/{nsn}/views", headers=h(tok)).json().get("identifiers", [])
        if tleft or vleft:
            print(f"   ❌ namespace {nsn} still has tables={len(tleft)} views={len(vleft)} — cannot delete")
            continue
        rns = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{nsn}", headers=h(tok))
        if rns.status_code not in (200,204):
            print(f"   ⚠️ namespace {nsn} delete: {rns.status_code} {rns.text[:100]}")

    # drop non-default catalog-roles (a catalog with extra roles is "not empty")
    for cr in requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles", headers=h(tok)).json().get("roles", []):
        crn = cr["name"] if isinstance(cr, dict) else cr
        if crn != "catalog_admin":
            rcr = requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{crn}", headers=h(tok))
            if rcr.status_code not in (200,204):
                print(f"   ⚠️ catalog-role {crn} delete: {rcr.status_code} {rcr.text[:80]}")
    rc = requests.delete(f"{BASE_MGMT}/catalogs/{cat}?purgeRequested=true", headers=h(tok))
    mc.delete_prefix(f"{cat}/", min_size=0)
    still = requests.get(f"{BASE_MGMT}/catalogs/{cat}", headers=h(tok)).status_code != 404
    if still:
        print(f"   ❌ catalog {cat} STILL EXISTS (delete status {rc.status_code}): {rc.text[:150]}")
        # show what's blocking
        leftover = requests.get(f"{BASE_CAT}/{cat}/namespaces", headers=h(tok)).json().get("namespaces", [])
        print(f"      remaining namespaces: {leftover}")
        return False
    return True

print("✅ force_delete_catalog ready")


✅ force_delete_catalog ready


## Test matrix — 8 cases across config × privilege × purgeRequested

We map which gate controls a view drop. Four dimensions interact:

```
purge-view-metadata-on-drop   catalog config (true/false)
drop-with-purge.enabled       catalog config (true/false)
?purgeRequested=              request param  (true/false/absent)
CATALOG_MANAGE_CONTENT        principal privilege (granted/not)
```

Because a grant can't be cleanly revoked mid-test, the runner uses **two catalogs**:
one WITH the grant and one WITHOUT. Each case runs against the right one.

```
C1  purge-meta=true  drop-purge=false  grant=NO   purgeReq=absent   ← original 403?
C2  purge-meta=true  drop-purge=false  grant=YES  purgeReq=absent   isolates privilege
C3  purge-meta=true  drop-purge=true   grant=NO   purgeReq=absent   isolates config
C4  purge-meta=true  drop-purge=true   grant=YES  purgeReq=absent   everything on
C5  purge-meta=true  drop-purge=true   grant=YES  purgeReq=false    explicit false
C6  purge-meta=false drop-purge=true   grant=YES  purgeReq=true     param vs config
C7  purge-meta=false drop-purge=true   grant=YES  purgeReq=absent   metadata-only baseline
C8  (drop an already-dropped view)                                  404 idempotency
```


In [3]:
# Two catalogs: one WITH CATALOG_MANAGE_CONTENT granted, one WITHOUT.
CAT_GRANT   = "view-test-grant"
CAT_NOGRANT = "view-test-nogrant"
NS = "vp-ns"

def setup_catalog(cat, grant):
    """Create a fresh catalog; grant CATALOG_MANAGE_CONTENT only if grant=True."""
    force_delete_catalog(cat)
    create_catalog(cat, token=tok, properties={
        "polaris.config.purge-view-metadata-on-drop":"true",
        "polaris.config.drop-with-purge.enabled":"true"})
    if grant:
        grant_privilege(catalog=cat, cr="catalog_admin",
                        privilege="CATALOG_MANAGE_CONTENT", token=tok)
    create_namespace(catalog=cat, ns=NS, token=tok)
    return cat

setup_catalog(CAT_GRANT,   grant=True)
setup_catalog(CAT_NOGRANT, grant=False)
print("✅ two catalogs ready: granted + non-granted")


✅ two catalogs ready: granted + non-granted


### Verify privileges BEFORE running the matrix

Confirms the actual identity + grants we're testing with, so the results are
interpretable. Specifically: which principal the token belongs to, and what
grants exist on each catalog's `catalog_admin` role. If the token is the **root
principal**, it may bypass privilege checks — which would explain why a
privilege-gated 403 doesn't appear for any matrix case.


In [4]:
# Who are we, and what does each catalog's catalog_admin role actually hold?
import base64, json as _json

# Decode the principal from the JWT (no verification — just to read the subject)
def token_principal(t):
    try:
        payload = t.split(".")[1]
        payload += "=" * (-len(payload) % 4)
        claims = _json.loads(base64.urlsafe_b64decode(payload))
        return claims.get("sub") or claims.get("principal") or claims.get("client_id") or claims
    except Exception as e:
        return f"(could not decode: {e})"

print("Token principal (subject):", token_principal(tok))
print(f"ROOT_CLIENT config: {ROOT_CLIENT}")
print()

# List grants on each test catalog's catalog_admin role
def show_grants(cat):
    r = requests.get(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/catalog_admin/grants",
                     headers=h(tok))
    if r.status_code != 200:
        print(f"  {cat}: cannot read grants ({r.status_code})")
        return []
    grants = r.json().get("grants", [])
    privs = [g.get("privilege") for g in grants]
    print(f"  {cat:20} catalog_admin grants: {privs if privs else '(none)'}")
    return privs

print("Catalog-role grants (what catalog_admin can do on each catalog):")
g_grant   = show_grants(CAT_GRANT)
g_nogrant = show_grants(CAT_NOGRANT)
print()

# Which principal-roles is our principal mapped to? (root is typically service_admin)
pr = requests.get(f"{BASE_MGMT}/principal-roles", headers=h(tok))
if pr.status_code == 200:
    print("Principal-roles that exist:", [r["name"] for r in pr.json().get("roles", [])])
print()

# Interpretation
print("="*64)
if "CATALOG_MANAGE_CONTENT" in g_grant and "CATALOG_MANAGE_CONTENT" not in g_nogrant:
    print("✅ Grant state is as intended:")
    print(f"   {CAT_GRANT}: HAS CATALOG_MANAGE_CONTENT")
    print(f"   {CAT_NOGRANT}: does NOT")
    print("   → if BOTH still allow the view drop (204), the token likely has")
    print("     root/service_admin authority that BYPASSES catalog-role grants.")
else:
    print("⚠️ Grant state not as expected — check setup:")
    print(f"   {CAT_GRANT} grants:   {g_grant}")
    print(f"   {CAT_NOGRANT} grants: {g_nogrant}")
print()
print("KEY CAVEAT: we authenticate as the ROOT principal here. Root commonly has")
print("service_admin, which bypasses per-catalog privilege checks. So 'no-grant'")
print("cases (C1/C3) may still succeed NOT because privilege doesn't gate view")
print("drops, but because ROOT ignores that gate. To truly test the privilege")
print("gate, use a NON-root principal (see optional C9).")


Token principal (subject): root
ROOT_CLIENT config: root

Catalog-role grants (what catalog_admin can do on each catalog):
  view-test-grant      catalog_admin grants: ['CATALOG_MANAGE_ACCESS', 'CATALOG_MANAGE_METADATA', 'CATALOG_MANAGE_CONTENT']
  view-test-nogrant    catalog_admin grants: ['CATALOG_MANAGE_ACCESS', 'CATALOG_MANAGE_METADATA']

Principal-roles that exist: ['service_admin']

✅ Grant state is as intended:
   view-test-grant: HAS CATALOG_MANAGE_CONTENT
   view-test-nogrant: does NOT
   → if BOTH still allow the view drop (204), the token likely has
     root/service_admin authority that BYPASSES catalog-role grants.

KEY CAVEAT: we authenticate as the ROOT principal here. Root commonly has
service_admin, which bypasses per-catalog privilege checks. So 'no-grant'
cases (C1/C3) may still succeed NOT because privilege doesn't gate view
drops, but because ROOT ignores that gate. To truly test the privilege
gate, use a NON-root principal (see optional C9).


In [5]:
results = {}

def make_view(cat, name):
    requests.post(f"{BASE_CAT}/{cat}/namespaces/{NS}/views", headers=h(tok),
        json={"name": name, "default-namespace": [NS],
              "schema":{"type":"struct","schema-id":0,
                        "fields":[{"id":1,"name":"id","type":"long","required":True}]},
              "view-version":{"version-id":1,"timestamp-ms":int(time.time()*1000),"schema-id":0,
                              "default-namespace":[NS],"summary":{"engine":"spark"},
                              "representations":[{"type":"sql","sql":"SELECT 1 AS id","dialect":"spark"}]},
              "location": f"s3a://{BUCKET}/{cat}/{NS}/{name}/"})

def cat_files(cat, name):
    return mc.list(f"{cat}/{NS}/{name}/")

def run_case(case, purge_meta, drop_purge, grant, purge_requested):
    """One case. grant selects which catalog; purge_requested True/False/None."""
    cat = CAT_GRANT if grant else CAT_NOGRANT
    # set the two config flags on the chosen catalog
    set_catalog_config(cat,
                       purge_view_metadata_on_drop=str(purge_meta).lower(),
                       drop_with_purge_enabled=str(drop_purge).lower())
    vname = f"{case}-{uuid.uuid4().hex[:6]}"
    make_view(cat, vname)
    before = len(cat_files(cat, vname))

    # build the drop URL with the right purgeRequested form
    url = f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}"
    if purge_requested is True:   url += "?purgeRequested=true"
    elif purge_requested is False: url += "?purgeRequested=false"
    # None → omit the param entirely
    r = requests.delete(url, headers=h(tok))

    gone = requests.get(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}",
                        headers=h(tok)).status_code == 404
    time.sleep(2)
    after = len(cat_files(cat, vname))

    pr = {True:"true", False:"false", None:"absent"}[purge_requested]
    results[case] = {"purge_meta":purge_meta, "drop_purge":drop_purge, "grant":grant,
                     "purge_req":pr, "status":r.status_code, "gone":gone,
                     "before":before, "after":after,
                     "err": r.text[:140] if r.status_code>=400 else ""}
    fo = ("orphaned" if after>=before and before>0 else
          "deleted"  if after==0 and before>0 else f"{before}->{after}")
    print(f"[{case}] meta={purge_meta} purge={drop_purge} grant={grant} req={pr}")
    print(f"   status={r.status_code}  gone={gone}  file={fo}"
          + (f"  ERR: {r.text[:90]}" if r.status_code>=400 else ""))
    return results[case]

print("✅ run_case ready")


✅ run_case ready


In [6]:
# C1 — purge-meta=true, drop-purge=false, NO grant, purgeReq absent  (original 403?)
run_case("C1", purge_meta=True,  drop_purge=False, grant=False, purge_requested=None)

[C1] meta=True purge=False grant=False req=absent
   status=403  gone=False  file=orphaned  ERR: {"error":{"message":"Unable to purge entity: C1-8eae53. To enable this feature, set the Po


{'purge_meta': True,
 'drop_purge': False,
 'grant': False,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'err': '{"error":{"message":"Unable to purge entity: C1-8eae53. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the'}

In [7]:
# C2 — purge-meta=true, drop-purge=false, WITH grant  (isolates privilege)
run_case("C2", purge_meta=True,  drop_purge=False, grant=True,  purge_requested=None)

[C2] meta=True purge=False grant=True req=absent
   status=403  gone=False  file=orphaned  ERR: {"error":{"message":"Unable to purge entity: C2-548b65. To enable this feature, set the Po


{'purge_meta': True,
 'drop_purge': False,
 'grant': True,
 'purge_req': 'absent',
 'status': 403,
 'gone': False,
 'before': 1,
 'after': 1,
 'err': '{"error":{"message":"Unable to purge entity: C2-548b65. To enable this feature, set the Polaris configuration DROP_WITH_PURGE_ENABLED or the'}

In [8]:
# C3 — purge-meta=true, drop-purge=true, NO grant  (isolates config)
run_case("C3", purge_meta=True,  drop_purge=True,  grant=False, purge_requested=None)

[C3] meta=True purge=True grant=False req=absent
   status=204  gone=True  file=orphaned


{'purge_meta': True,
 'drop_purge': True,
 'grant': False,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'err': ''}

In [9]:
# C4 — purge-meta=true, drop-purge=true, WITH grant  (everything on)
run_case("C4", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=None)

[C4] meta=True purge=True grant=True req=absent
   status=204  gone=True  file=orphaned


{'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'err': ''}

In [10]:
# C5 — explicit purgeRequested=false (vs absent in C4)
run_case("C5", purge_meta=True,  drop_purge=True,  grant=True,  purge_requested=False)

[C5] meta=True purge=True grant=True req=false
   status=204  gone=True  file=orphaned


{'purge_meta': True,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'false',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'err': ''}

In [11]:
# C6 — purgeRequested=true but purge-meta=false  (param vs config conflict)
run_case("C6", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=True)

[C6] meta=False purge=True grant=True req=true
   status=204  gone=True  file=orphaned


{'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'true',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'err': ''}

In [12]:
# C7 — purge-meta=false  (metadata-only baseline)
run_case("C7", purge_meta=False, drop_purge=True,  grant=True,  purge_requested=None)

[C7] meta=False purge=True grant=True req=absent
   status=204  gone=True  file=orphaned


{'purge_meta': False,
 'drop_purge': True,
 'grant': True,
 'purge_req': 'absent',
 'status': 204,
 'gone': True,
 'before': 1,
 'after': 1,
 'err': ''}

In [13]:
# C8 — idempotency: drop an already-dropped view (expect 404)
cat = CAT_GRANT
vname = f"C8-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)
r1 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=h(tok))
r2 = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=h(tok))
results["C8"] = {"purge_meta":"-","drop_purge":"-","grant":True,"purge_req":"absent",
                 "status":r2.status_code,"gone":True,"before":0,"after":0,
                 "err":r2.text[:140] if r2.status_code>=400 else ""}
print(f"[C8] first drop={r1.status_code}, second drop={r2.status_code} (expect 404 idempotent)")

[C8] first drop=204, second drop=404 (expect 404 idempotent)


## C9 — the real privilege test: a NON-ROOT principal

C1–C8 all ran as **root** (service_admin), which bypasses catalog-role privilege
checks — so they couldn't trigger a privilege 403. C9 creates a fresh principal
that has only enough rights to SEE the catalog but **NOT** `CATALOG_MANAGE_CONTENT`,
authenticates as that principal, and tries to drop a view.

```
If C9 → 403  →  the privilege gate is REAL; root was bypassing it.
                The original 403 was a non-root principal lacking the privilege.
If C9 → 204  →  view drop needs no special privilege even for non-root.
```


In [14]:
# C9 — create a limited non-root principal and drop a view as that principal
P9   = "view-limited-principal"
PR9  = "view-limited-prole"
CR9  = "view-limited-crole"
cat  = CAT_NOGRANT   # this catalog's catalog_admin lacks CATALOG_MANAGE_CONTENT anyway

# clean any leftovers from a prior run
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))

# 1. create principal → capture its client credentials
rp = create_principal(P9, token=tok)
creds = rp.json()
client_id     = creds.get("credentials", {}).get("clientId") or creds.get("clientId")
client_secret = creds.get("credentials", {}).get("clientSecret") or creds.get("clientSecret")
print(f"created principal {P9}: client_id={client_id}")

# 2. principal-role + catalog-role, give the catalog-role ONLY limited privileges
create_principal_role(PR9, token=tok)
create_catalog_role(catalog=cat, name=CR9, token=tok)
assign_catalog_role_to_principal_role(catalog=cat, pr=PR9, cr=CR9, token=tok)
assign_principal_role_to_principal(principal=P9, pr=PR9, token=tok)
# grant ONLY enough to use the catalog + drop a view, but NOT manage-content/purge
for priv in ["CATALOG_MANAGE_METADATA", "TABLE_LIST", "VIEW_LIST", "VIEW_CREATE", "VIEW_DROP"]:
    grant_privilege(catalog=cat, cr=CR9, privilege=priv, token=tok)
print(f"granted limited privileges to {CR9} (NO CATALOG_MANAGE_CONTENT)")

# 3. get a token AS the limited principal
def principal_token(cid, csec):
    r = get_token(client_id=cid, client_secret=csec)
    return r.json().get("access_token") if r.status_code == 200 else None

ltok = principal_token(client_id, client_secret)
print(f"limited-principal token obtained: {ltok is not None}")

# 4. create a view (as root, to be sure it exists), then DROP as the limited principal
set_catalog_config(cat, purge_view_metadata_on_drop="true", drop_with_purge_enabled="true")
vname = f"C9-{uuid.uuid4().hex[:6]}"
make_view(cat, vname)   # created with root

if ltok:
    rh = {"Authorization": f"Bearer {ltok}", "Polaris-Realm": REALM, "Content-Type": "application/json"}
    r = requests.delete(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=rh)
    print(f"\n[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): {r.status_code}")
    print(f"     {r.text[:200]}")
    results["C9"] = {"purge_meta":True,"drop_purge":True,"grant":"non-root-limited",
                     "purge_req":"absent","status":r.status_code,
                     "gone": requests.get(f"{BASE_CAT}/{cat}/namespaces/{NS}/views/{vname}", headers=h(tok)).status_code==404,
                     "before":1,"after":1,"err":r.text[:140] if r.status_code>=400 else ""}
    print()
    if r.status_code == 403:
        print("✅ C9 = 403 → PRIVILEGE GATE CONFIRMED.")
        print("   A non-root principal without CATALOG_MANAGE_CONTENT CANNOT drop the view.")
        print("   → root (service_admin) was bypassing this gate in C1–C8.")
        print("   → the original 403 was a non-root principal lacking the privilege.")
    elif r.status_code in (200,204):
        print("ℹ️ C9 = 204 → even a limited non-root principal can drop a view.")
        print("   → VIEW_DROP alone suffices; the 403 came from something else.")
    else:
        print(f"ℹ️ C9 = {r.status_code} — inspect the error body above.")
else:
    print("⚠️ could not obtain limited-principal token; check principal creation")


created principal view-limited-principal: client_id=34491b1673e6d174
granted limited privileges to view-limited-crole (NO CATALOG_MANAGE_CONTENT)
limited-principal token obtained: True

[C9] DROP as NON-ROOT (no CATALOG_MANAGE_CONTENT): 204
     

ℹ️ C9 = 204 → even a limited non-root principal can drop a view.
   → VIEW_DROP alone suffices; the 403 came from something else.


In [15]:
# C9 cleanup — remove the limited principal, its roles, AND its catalog-role
requests.delete(f"{BASE_MGMT}/principals/{P9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/principal-roles/{PR9}", headers=h(tok))
requests.delete(f"{BASE_MGMT}/catalogs/{cat}/catalog-roles/{CR9}", headers=h(tok))  # ← was leaking
print("✅ C9 principal + principal-role + catalog-role removed")


✅ C9 principal + principal-role + catalog-role removed


## Truth table — observed results

In [16]:
print("="*78)
print(f"VIEW DROP — 8-case matrix (Polaris {POLARIS_VERSION})")
print("="*78)
print(f"{'case':5}{'purge-meta':11}{'drop-purge':11}{'grant':7}{'purgeReq':9}{'status':7}{'file':10}{'gone'}")
print("-"*78)
order = ["C1","C2","C3","C4","C5","C6","C7","C8","C9"]
for cse in order:
    r = results.get(cse)
    if not r:
        print(f"{cse:5}(not run)"); continue
    fo = ("orphaned" if r["after"]>=r["before"] and r["before"]>0 else
          "deleted"  if r["after"]==0 and r["before"]>0 else "-")
    print(f"{cse:5}{str(r['purge_meta']):11}{str(r['drop_purge']):11}"
          f"{str(r['grant']):7}{r['purge_req']:9}{str(r['status']):7}{fo:10}{r['gone']}")
    if r["err"]:
        print(f"      └─ {r['err'][:90]}")

print()
print("="*78)
print("WHAT EACH GATE DOES (derived from the cases above)")
print("="*78)

def st(c): return results.get(c,{}).get("status")

# Privilege gate: C1/C3 (no grant) vs C2/C4 (grant)
print("\n1. PRIVILEGE gate (CATALOG_MANAGE_CONTENT):")
print(f"   C1 no-grant={st('C1')}  vs  C2 grant={st('C2')}   (drop-purge=false)")
print(f"   C3 no-grant={st('C3')}  vs  C4 grant={st('C4')}   (drop-purge=true)")
if st("C1")==403 or st("C3")==403:
    print("   → 403 appears WITHOUT the grant → the 403 is a PRIVILEGE error.")
    print("     The original '403' was a missing CATALOG_MANAGE_CONTENT, not the config.")
elif st("C1") in (200,204) and st("C2") in (200,204):
    print("   → no 403 even without grant → privilege is not the blocking gate here.")

# Config gate: compare drop-purge true vs false at same grant
print("\n2. CONFIG gate (drop-with-purge.enabled), holding grant + purge-meta=true:")
print(f"   C2 drop-purge=false={st('C2')}  vs  C4 drop-purge=true={st('C4')}")
if st("C2")==st("C4"):
    print("   → same result → drop-with-purge.enabled does NOT change the view-drop status.")
else:
    print("   → differs → drop-with-purge.enabled DOES gate the view drop.")

# purgeRequested param
print("\n3. purgeRequested param:")
print(f"   C4 absent={st('C4')}  vs  C5 explicit-false={st('C5')}")
print(f"   C6 (purgeReq=true, purge-meta=false) status={st('C6')}, "
      f"file={'orphaned' if results.get('C6',{}).get('after',0)>=results.get('C6',{}).get('before',1) else 'deleted'}")

print("\n4. Baseline + idempotency:")
print(f"   C7 metadata-only (purge-meta=false) status={st('C7')}")
print(f"   C8 double-drop status={st('C8')} (expect 404)")

print("\n5. PRIVILEGE GATE — the decisive test (C9, non-root principal):")
c9 = results.get("C9", {})
if c9:
    print(f"   C9 (non-root, no CATALOG_MANAGE_CONTENT) status = {c9.get('status')}")
    if c9.get("status") == 403:
        print("   → ✅ PRIVILEGE GATE IS REAL. Root (service_admin) bypassed it in C1-C8.")
        print("        The original 403 = a non-root principal lacking the privilege.")
    elif c9.get("status") in (200,204):
        print("   → view drop needs no special privilege even for non-root; 403 came")
        print("        from elsewhere (different entity, version, or realm).")
else:
    print("   (C9 not run)")
print("\nNOTE: where 'file=orphaned', the drop succeeded but the metadata file was")
print("      NOT deleted (no-STS cleanup-task failure, Polaris #379).")


VIEW DROP — 8-case matrix (Polaris 1.3.0)
case purge-meta drop-purge grant  purgeReq status file      gone
------------------------------------------------------------------------------
C1   True       False      False  absent   403    orphaned  False
      └─ {"error":{"message":"Unable to purge entity: C1-8eae53. To enable this feature, set the Po
C2   True       False      True   absent   403    orphaned  False
      └─ {"error":{"message":"Unable to purge entity: C2-548b65. To enable this feature, set the Po
C3   True       True       False  absent   204    orphaned  True
C4   True       True       True   absent   204    orphaned  True
C5   True       True       True   false    204    orphaned  True
C6   False      True       True   true     204    orphaned  True
C7   False      True       True   absent   204    orphaned  True
C8   -          -          True   absent   404    -         True
      └─ {"error":{"message":"View does not exist: vp-ns.C8-d85e80","type":"NoSuchViewExcept

## Cleanup

In [17]:
for cat in [CAT_GRANT, CAT_NOGRANT]:
    force_delete_catalog(cat)
print("✅ cleaned up both test catalogs")

✅ cleaned up both test catalogs
